# ML-08 — Capstone Modeling Lane

Refresh / Content Opportunity Scoring. Builds on Week 1–4 (research question, task framing,
leakage-safe data contract, Week 4 rule baseline).


In [ ]:
import duckdb
con = duckdb.connect(database=':memory:', read_only=False)
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{BASE}/fact_content_daily_performance/**/*.parquet"
FACT_QUERY = f"{BASE}/fact_content_query_90d/**/*.parquet"
DIM_CLIENTS = f"{BASE}/dim_clients/**/*.parquet"
DIM_CONTENT = f"{BASE}/dim_content/**/*.parquet"


## 1. Method choice and why

**Logistic regression, class-balanced.** Three honest reasons, not a default:

- The unit of analysis after aggregation is one row per eligible content item — tens of
  thousands of rows, not millions. A linear model is sized to match the data, not chosen to
  look impressive.
- Coefficients are directly readable as reason codes ("this item is flagged mainly because
  `avg_position_prev30` is high relative to its trailing baseline"). A decision-support tool
  needs to say *why*, not just output a probability.
- `class_weight='balanced'` handles the fact that most content isn't declining in any given
  30-day window, without synthetic oversampling that would be harder to explain honestly in
  the paper.

Checked below: how imbalanced the label actually is, before trusting that choice.


In [ ]:
eligible_clients_sql = f"""
    SELECT client_hash_id FROM read_parquet('{DIM_CLIENTS}')
    WHERE access_profile = 'gsc_and_ga4'
"""

WINDOW_START = "2026-04-02"           # fact_content_query_90d.window_start, fixed for this release
DAILY_FEATURE_START = "2025-01-02"    # generous trailing lookback; content_age_days shows how much of it each item actually had
MIN_PREV30_IMPRESSIONS = 20
POSITION_WORSENING_THRESHOLD = 3.0
IMPRESSIONS_DROP_THRESHOLD = 0.20
CLICK_DROP_THRESHOLD = 0.20

eligible_content_sql = f"""
WITH eligible_clients AS ({eligible_clients_sql})
SELECT
    client_hash_id, content_hash_id, content_created_date, content_updated_date,
    main_intent, search_volume, competition, cpc, backlinks, word_count, category_count,
    last_optimized_date, optimization_eligible_date,
    DATE_DIFF('day', content_created_date, DATE '{WINDOW_START}') AS content_age_days,
    (content_updated_date >= DATE '{WINDOW_START}') AS updated_during_window,
    (last_optimized_date IS NOT NULL AND last_optimized_date >= DATE '{WINDOW_START}') AS optimized_during_window
FROM read_parquet('{DIM_CONTENT}')
WHERE is_published = true AND is_deleted = false
  AND content_created_date < DATE '{WINDOW_START}'
  AND client_hash_id IN (SELECT client_hash_id FROM eligible_clients)
"""
eligible_content = con.sql(eligible_content_sql).df()

daily_feat_sql = f"""
SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS impr_daily_trailing,
    SUM(gsc_clicks) AS clicks_daily_trailing,
    AVG(gsc_avg_position) AS avgpos_daily_trailing,
    COUNT(*) AS days_observed_daily_trailing
FROM read_parquet('{FACT_DAILY}')
WHERE report_date >= '{DAILY_FEATURE_START}' AND report_date < '{WINDOW_START}'
GROUP BY 1,2
"""
daily_feat = con.sql(daily_feat_sql).df()

query_agg_sql = f"""
SELECT client_hash_id, content_hash_id,
    SUM(impressions_prev30) AS impr_prev30,
    SUM(clicks_prev30) AS clicks_prev30,
    SUM(avg_position_prev30 * impressions_prev30) / NULLIF(SUM(impressions_prev30), 0) AS avgpos_prev30_w,
    SUM(impressions_last30) AS impr_last30,
    SUM(clicks_last30) AS clicks_last30,
    SUM(avg_position_last30 * impressions_last30) / NULLIF(SUM(impressions_last30), 0) AS avgpos_last30_w
FROM read_parquet('{FACT_QUERY}')
GROUP BY 1,2
"""
query_agg = con.sql(query_agg_sql).df()

df = (eligible_content
      .merge(daily_feat, on=["client_hash_id", "content_hash_id"], how="left")
      .merge(query_agg, on=["client_hash_id", "content_hash_id"], how="inner"))
df = df[df["impr_prev30"] >= MIN_PREV30_IMPRESSIONS].copy()
df[["impr_daily_trailing","clicks_daily_trailing","days_observed_daily_trailing"]] = \
    df[["impr_daily_trailing","clicks_daily_trailing","days_observed_daily_trailing"]].fillna(0)

confounded = df["updated_during_window"] | df["optimized_during_window"]
df = df[~confounded].copy()

impr_drop = (df["impr_prev30"] - df["impr_last30"]) / df["impr_prev30"].replace(0, float("nan"))
click_drop = (df["clicks_prev30"] - df["clicks_last30"]) / df["clicks_prev30"].replace(0, float("nan"))
worsened = (df["avgpos_last30_w"] - df["avgpos_prev30_w"]) >= POSITION_WORSENING_THRESHOLD
dropped = (impr_drop >= IMPRESSIONS_DROP_THRESHOLD) | (click_drop >= CLICK_DROP_THRESHOLD)
df["is_declining"] = (worsened & dropped).astype(int)

print("Rows after eligibility + confound exclusion:", len(df))
print(df["is_declining"].value_counts(normalize=True))


## 2. Split design

**Grouped by `client_hash_id`, not time-aware in the rolling sense.** The label itself
(`last30` vs `prev30`) already comes from one fixed cutoff shared by every content item — the
warehouse release only has one `window_start`/`window_end`, so there's no meaningful rolling
time split to do here; a single snapshot doesn't have multiple time folds.

The real leakage risk at this grain is **client-level correlation**, not date order — content
items from the same client share traffic patterns, seasonality, and any client-side site
changes. A plain random row split would let the model partly learn client-specific quirks in
training and coast on that in test. `GroupShuffleSplit` on `client_hash_id` keeps each client's
content entirely on one side of the split — confirmed below, not just assumed.


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df = pd.get_dummies(df, columns=["main_intent"], prefix="intent", dummy_na=True)
intent_cols = [c for c in df.columns if c.startswith("intent_")]

feature_cols = [
    "avgpos_daily_trailing", "impr_daily_trailing", "clicks_daily_trailing", "days_observed_daily_trailing",
    "avgpos_prev30_w", "impr_prev30", "clicks_prev30",
    "content_age_days", "search_volume", "competition", "cpc", "backlinks", "word_count", "category_count",
] + intent_cols

work = df.dropna(subset=[c for c in feature_cols if not c.startswith("intent_")] + ["is_declining"]).reset_index(drop=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(work, groups=work["client_hash_id"]))
train, test = work.iloc[train_idx].copy(), work.iloc[test_idx].copy()

assert set(train["client_hash_id"]) & set(test["client_hash_id"]) == set(), "Client leakage across split!"
print(f"Train: {len(train)} rows / {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test)} rows / {test['client_hash_id'].nunique()} clients")


## 3. Train + compare vs my baseline

Same test set, same metric (**Precision@50**) as Week 4. Baseline is recomputed here on this
exact held-out set so it's a fair comparison, not a number pulled from a different sample.


In [ ]:
test["baseline_score"] = test["avgpos_prev30_w"] - test["avgpos_daily_trailing"]
baseline_top50 = test.sort_values("baseline_score", ascending=False).head(50)
baseline_p50 = baseline_top50["is_declining"].mean()

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
X_test = scaler.transform(test[feature_cols])
model = LogisticRegression(class_weight="balanced", max_iter=1000).fit(X_train, train["is_declining"])
test["model_score"] = model.predict_proba(X_test)[:, 1]
model_top50 = test.sort_values("model_score", ascending=False).head(50)
model_p50 = model_top50["is_declining"].mean()

comparison = pd.DataFrame({
    "method": ["Week-4 baseline (position-worsening rule)", "Logistic regression (this notebook)"],
    "precision_at_50": [baseline_p50, model_p50],
    "positive_rate_in_test": [test["is_declining"].mean()] * 2,
    "n_test": [len(test)] * 2,
})
comparison


## 4. Errors and interpretation

Where the model is wrong, and what it leans on — in numbers, not a bigger metric table.


In [ ]:
coefs = pd.Series(model.coef_[0], index=feature_cols).sort_values(key=abs, ascending=False)
print("Top 5 features by |coefficient| (standardized):")
print(coefs.head(5))

test["predicted"] = (test["model_score"] >= 0.5).astype(int)
fp = test[(test["predicted"] == 1) & (test["is_declining"] == 0)]
fn = test[(test["predicted"] == 0) & (test["is_declining"] == 1)]
tp = test[(test["predicted"] == 1) & (test["is_declining"] == 1)]

print(f"\nFalse positives: {len(fp)}, False negatives: {len(fn)}, True positives: {len(tp)}")
print("\nAvg trailing impressions -- FP vs TP (is the model chasing noisy high-volume pages?):")
print(f"  false positives: {fp['impr_daily_trailing'].mean():.1f}")
print(f"  true positives:  {tp['impr_daily_trailing'].mean():.1f}")
print("\nAvg position_prev30 -- FN vs TP (is the model missing slow, already-bad-position declines?):")
print(f"  false negatives: {fn['avgpos_prev30_w'].mean():.1f}")
print(f"  true positives:  {tp['avgpos_prev30_w'].mean():.1f}")


**Reading (fill in with your actual numbers once run):** the model leans most on the two
features printed above. If false positives run higher on trailing impressions than true
positives, the model is over-flagging high-traffic pages with normal week-to-week noise. If
false negatives sit at a worse `avgpos_prev30_w` than true positives, the model is under-flagging
content that was already declining slowly before `prev30` even started — a real limitation worth
stating plainly in the paper, not smoothing over.


## Self-check

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere — only `client_*`/`content_*` hashes
- [ ] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to `work/notebooks/`, repo URL submitted on the card
